# DriftLab — Master Project Notebook

One evolving notebook for exploratory analysis, training, model comparison and future monitoring. **Run from top to bottom.** The dataset should be at `/content/ai4i2020.csv` in Colab or `data/raw/ai4i2020.csv` locally.

**Experiment policy:** Training data fits model and preprocessors; validation selects models and thresholds; test is held back until the final selection.

In [1]:
pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Section 1 — Dataset understanding (Day 1)

**Why:** Inspect schema, completeness, class imbalance and possible leakage *before* learning patterns.

In [2]:
# Uncomment only if needed:
# %pip install numpy pandas matplotlib scikit-learn torch

from pathlib import Path
from copy import deepcopy
import random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('sklearn:', sklearn.__version__, '| torch:', torch.__version__)
print('device: cpu (small tabular dataset)')

ImportError: DLL load failed while importing _dop: An Application Control policy has blocked this file.

In [5]:
paths = [Path('/content/ai4i2020.csv'), Path('../data/raw/ai4i2020.csv'),
         Path('data/raw/ai4i2020.csv'), Path('ai4i2020.csv')]
path = next((p for p in paths if p.is_file()), None)
if path is None:
    raise FileNotFoundError('Upload ai4i2020.csv in Colab or put it under data/raw/.')
df = pd.read_csv(path)
num_cols = ['Air temperature [K]', 'Process temperature [K]',
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
cat_cols = ['Type']
features = cat_cols + num_cols
target = 'Machine failure'
for c in features + [target]:
    assert c in df.columns, f'Missing column: {c}'
assert df[features + [target]].notna().all().all()
X = df[features].copy()
y = df[target].astype(int).copy()
assert set(y.unique()) == {0, 1}
print('Rows:', len(df), '| positive share:', round(y.mean(), 4))
display(df[features + [target]].head())

Rows: 10000 | positive share: 0.0339


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [11]:
print('Shape:', df.shape)
print('Missing values by column:')
print(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Target distribution:')
print(y.value_counts().sort_index())
print('Target proportions:')
print(y.value_counts(normalize=True).sort_index().round(4))
display(df[num_cols].describe().T)
display(df.groupby(target)[num_cols].agg(['mean', 'median', 'std']).round(2))
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
y.value_counts().sort_index().plot.bar(ax=axs[0], title='Machine failure class counts')
axs[0].set_xticklabels(['Normal (0)', 'Failure (1)'], rotation=0)
axs[0].set_ylabel('Observations')
axs[1].boxplot([df.loc[y == 0, 'Torque [Nm]'], df.loc[y == 1, 'Torque [Nm]']], tick_labels=['Normal', 'Failure'])
axs[1].set_title('Torque distribution by failure class')
axs[1].set_ylabel('Nm')
plt.tight_layout(); plt.show()

NameError: name 'df' is not defined

**Leakage note:** `Machine failure` is the target. The five failure-mode indicators (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) are outcome-related labels, not predictive inputs. `UDI` and `Product ID` are excluded from this baseline. Only `Type` and five numeric measurements enter `X`. This synthetic dataset predicts observed failure status, **not time to future failure**.

## Section 2 — Split and train Logistic Regression (Day 2)

**Why:** Evaluate against unseen data, with no preprocessing fitted on validation or test samples.

In [8]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15 / 0.85,
    stratify=y_trainval, random_state=SEED
)
assert [len(X_train), len(X_val), len(X_test)] == [7000, 1500, 1500]
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
for name, labels in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    print(f'{name}: n={len(labels):,}, failure prevalence={labels.mean():.3%}')

ImportError: DLL load failed while importing _dop: An Application Control policy has blocked this file.

## 2 — Preprocessing (train only)

**Why:** Logistic Regression and neural networks benefit from scaling. One-hot encoding converts `Type` into indicator columns without inventing a numerical order. Trees do not require scaling, but we keep the same representation for easy comparison.

**Key:** Scaler and encoder learn only from `X_train`; validation and test rows are transformed using the training-derived rules.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def make_preprocessor():
    try:
        onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown='ignore', sparse=False)
    return ColumnTransformer([
        ('numeric', StandardScaler(), num_cols),
        ('categorical', onehot, cat_cols),
    ], remainder='drop')

# Each sklearn model owns its own preprocessing pipeline.
logreg = Pipeline([('preprocess', make_preprocessor()),
                   ('model', LogisticRegression(max_iter=2000, random_state=SEED))])
logreg.fit(X_train, y_train)

# Fit a separate transformer, ONLY on the train set, for PyTorch.
torch_preprocessor = make_preprocessor()
train_np = np.asarray(torch_preprocessor.fit_transform(X_train), dtype=np.float32)
val_np = np.asarray(torch_preprocessor.transform(X_val), dtype=np.float32)
# Test transform intentionally deferred until final evaluation.
print('Processed features:', train_np.shape[1])
assert train_np.shape[1] == val_np.shape[1]
assert np.isfinite(train_np).all() and np.isfinite(val_np).all()

ImportError: The `scipy` install you are using seems to be broken, (extension modules cannot be imported), please try reinstalling.

### Logistic Regression baseline and evaluation helpers

In [9]:
from sklearn.metrics import (average_precision_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             PrecisionRecallDisplay)

def choose_threshold(y_true, prob):
    rows = []
    for t in np.arange(0.05, 0.96, 0.05):
        pred = (prob >= t).astype(int)
        rows.append({'threshold': float(t),
                     'precision': precision_score(y_true, pred, zero_division=0),
                     'recall': recall_score(y_true, pred, zero_division=0),
                     'f1': f1_score(y_true, pred, zero_division=0)})
    scores = pd.DataFrame(rows)
    best = scores.sort_values(['f1', 'recall', 'threshold'],
                              ascending=[False, False, True]).iloc[0]
    return float(best['threshold']), scores

def summarize(name, y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    return {'model': name, 'threshold': threshold,
            'average_precision': average_precision_score(y_true, prob),
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'f1': f1_score(y_true, pred, zero_division=0)}

val_results = {}
val_probs = {}
val_thresholds = {}

logreg_prob = logreg.predict_proba(X_val)[:, 1]
logreg_t, _ = choose_threshold(y_val, logreg_prob)
val_probs['Logistic Regression'] = logreg_prob
val_thresholds['Logistic Regression'] = logreg_t
val_results['Logistic Regression'] = summarize('Logistic Regression', y_val, logreg_prob, logreg_t)
print(val_results['Logistic Regression'])

{'model': 'Logistic Regression', 'threshold': 0.25, 'average_precision': 0.3565343390600447, 'precision': 0.45454545454545453, 'recall': 0.39215686274509803, 'f1': 0.42105263157894735}


**Threshold selection:** Compare model versions using validation Average Precision; for each model, choose an F1-based classification threshold using validation only. Model selection and threshold optimization both use validation, so the held-out test is reserved for later.

## Section 3 — Random Forest experiments (Day 3)

**Why:** Trees can represent nonlinear interactions that a plain logistic model may miss.

In [1]:
rf_trials = [
    {'n_estimators': 100, 'max_depth': 6, 'min_samples_leaf': 3},
    {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 3},
    {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 5},
]
rf_experiments = []
rf_models = []
for config in rf_trials:
    rf = Pipeline([('preprocess', make_preprocessor()),
                   ('model', RandomForestClassifier(**config, random_state=SEED, n_jobs=-1))])
    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_val)[:, 1]
    score = average_precision_score(y_val, proba)
    rf_experiments.append({**config, 'val_average_precision': score})
    rf_models.append(rf)
rf_table = pd.DataFrame(rf_experiments).sort_values('val_average_precision', ascending=False)
display(rf_table)
rf_best_idx = int(np.argmax([r['val_average_precision'] for r in rf_experiments]))
rf_best = rf_models[rf_best_idx]
rf_prob = rf_best.predict_proba(X_val)[:, 1]
rf_t, _ = choose_threshold(y_val, rf_prob)
val_probs['Random Forest'] = rf_prob
val_thresholds['Random Forest'] = rf_t
val_results['Random Forest'] = summarize('Random Forest', y_val, rf_prob, rf_t)
print('Winning RF config:', rf_trials[rf_best_idx])

NameError: name 'Pipeline' is not defined

## Section 4 — PyTorch multilayer perceptron (Day 3)

**Why:** Learn neural network layers, logits, gradients and early stopping with direct control of the training loop.

In [ ]:
class FailureMLP(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)  # one logit for each row

mlp = FailureMLP(train_np.shape[1])
print(mlp)
print('Trainable parameters:', sum(p.numel() for p in mlp.parameters() if p.requires_grad))

In [ ]:
train_x = torch.from_numpy(train_np)
train_y = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
val_x = torch.from_numpy(val_np)
val_y = torch.tensor(y_val.to_numpy(), dtype=torch.float32)
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=128, shuffle=True)

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001, weight_decay=1e-4)

max_epochs = 100
patience = 12
best_loss = float('inf')
best_weights = None
stale_epochs = 0
history = {'train_loss': [], 'val_loss': []}

for epoch in range(max_epochs):
    mlp.train()
    total_loss = 0.0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        logits = mlp(batch_x)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_x)
    train_loss = total_loss / len(train_x)

    mlp.eval()
    with torch.no_grad():
        validation_loss = loss_fn(mlp(val_x), val_y).item()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(validation_loss)

    if validation_loss < best_loss - 1e-5:
        best_loss = validation_loss
        best_weights = deepcopy(mlp.state_dict())
        stale_epochs = 0
    else:
        stale_epochs += 1
    if stale_epochs >= patience:
        print(f'Early stopping at epoch {epoch + 1}')
        break

mlp.load_state_dict(best_weights)
print('Best validation loss:', round(best_loss, 4))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train loss')
plt.plot(history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Weighted BCEWithLogitsLoss')
plt.title('MLP learning curves')
plt.legend()
plt.tight_layout()
plt.show()

mlp.eval()
with torch.no_grad():
    mlp_prob = torch.sigmoid(mlp(val_x)).numpy()
mlp_t, _ = choose_threshold(y_val, mlp_prob)
val_probs['PyTorch MLP'] = mlp_prob
val_thresholds['PyTorch MLP'] = mlp_t
val_results['PyTorch MLP'] = summarize('PyTorch MLP', y_val, mlp_prob, mlp_t)

## Section 5 — Validation comparison and selection

**Why:** Compare all three using the *same* validation set before evaluating the test set.

In [ ]:
comparison = pd.DataFrame(val_results.values()).sort_values(
    'average_precision', ascending=False).reset_index(drop=True)
display(comparison.round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, prob in val_probs.items():
    PrecisionRecallDisplay.from_predictions(y_val, prob, name=name, ax=axes[0])
axes[0].set_title('Validation precision–recall curves')
axes[1].bar(comparison['model'], comparison['average_precision'])
axes[1].set_ylabel('Validation average precision')
axes[1].set_title('Threshold-independent comparison')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
winning_name = str(comparison.iloc[0]['model'])
winning_threshold = val_thresholds[winning_name]
print('Selected model:', winning_name)
print('Selected threshold:', round(winning_threshold, 2))
print('Selection based on validation Average Precision only.')

## Section 6 — Final held-out test evaluation

**Run this once, only after choosing the model and threshold.**

In [ ]:
if winning_name == 'Logistic Regression':
    test_prob = logreg.predict_proba(X_test)[:, 1]
elif winning_name == 'Random Forest':
    test_prob = rf_best.predict_proba(X_test)[:, 1]
else:
    test_np = np.asarray(torch_preprocessor.transform(X_test), dtype=np.float32)
    mlp.eval()
    with torch.no_grad():
        test_prob = torch.sigmoid(mlp(torch.from_numpy(test_np))).numpy()

test_summary = summarize(winning_name, y_test, test_prob, winning_threshold)
display(pd.DataFrame([test_summary]).round(4))
test_pred = (test_prob >= winning_threshold).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, labels=[0, 1], colorbar=False)
plt.title(f'Held-out test confusion matrix: {winning_name}')
plt.tight_layout()
plt.show()

## Section 7 — Engineering reflection

1. Why do trees not require scaling?
2. Why do we use `BCEWithLogitsLoss` instead of sigmoid followed by BCE?
3. Which false errors matter more for predictive maintenance, and why?
4. Why would validation Average Precision differ from F1 at a selected threshold?
5. Explain training, validation, and test separation in your own words.

### Future sections
- Day 4: Serialization and FastAPI inference
- Day 5: Drift monitoring, model registry and controlled retraining

Section 8 — Day 4: Export your selected trained model
What: Store the fitted estimator, its preprocessing, feature contract, selected threshold and model metadata.
Why: Production inference must apply exactly the transformations used during training. The threshold chosen on validation data is part of the deployed decision rule.
Run Sections 1–7 first. Do not reselect a model or threshold using the test set. Models exported here are trusted local artifacts only; joblib must not load files from unknown sources.

In [ ]:
from pathlib import Path
import json, joblib, zipfile

version = 'v1'
artifact_dir = Path('models') / version
artifact_dir.mkdir(parents=True, exist_ok=True)

assert 'winning_name' in globals() and 'winning_threshold' in globals()
meta = {
    'name': winning_name,
    'kind': 'torch' if winning_name == 'PyTorch MLP' else 'sklearn',
    'threshold': float(winning_threshold),
    'features': list(features),
    'input_features': int(train_np.shape[1]),
    'validation_average_precision': float(val_results[winning_name]['average_precision']),
    'sklearn_version': sklearn.__version__,
    'torch_version': torch.__version__,
    'training_rows': len(X_train),
    'note': 'AI4I synthetic observations; predictions describe recorded failure state, not future time-to-failure.'
}
if winning_name == 'PyTorch MLP':
    torch.save(mlp.cpu().state_dict(), artifact_dir / 'model.pt')
    joblib.dump(torch_preprocessor, artifact_dir / 'preprocessor.joblib')
else:
    trained_pipeline = logreg if winning_name == 'Logistic Regression' else rf_best
    joblib.dump(trained_pipeline, artifact_dir / 'model.joblib')
(artifact_dir / 'metadata.json').write_text(json.dumps(meta, indent=2))
Path('models/active.json').write_text(json.dumps({'version':version}, indent=2))
print('Exported:', winning_name, '| threshold:', winning_threshold)


Section 8.1 — Verify a reload matches the notebook
What: Compare predictions made before and after saving. Why: A file existing is not enough; loaded inference must reproduce trained inference.

In [ ]:
from sklearn.metrics import average_precision_score
sample_rows = X_val.iloc[:5].copy()
if meta['kind'] == 'sklearn':
    loaded = joblib.load(artifact_dir / 'model.joblib')
    before = trained_pipeline.predict_proba(sample_rows)[:, 1]
    after = loaded.predict_proba(sample_rows)[:, 1]
else:
    from torch import nn
    restored_preprocessor = joblib.load(artifact_dir / 'preprocessor.joblib')
    restored = FailureMLP(meta['input_features'])
    restored.load_state_dict(torch.load(artifact_dir / 'model.pt', map_location='cpu', weights_only=True))
    restored.eval()
    mlp.eval()
    with torch.inference_mode():
        original_features = torch.from_numpy(np.asarray(torch_preprocessor.transform(sample_rows),dtype=np.float32))
        loaded_features = torch.from_numpy(np.asarray(restored_preprocessor.transform(sample_rows),dtype=np.float32))
        before = torch.sigmoid(mlp(original_features)).numpy()
        after = torch.sigmoid(restored(loaded_features)).numpy()
np.testing.assert_allclose(before, after, atol=1e-6, rtol=1e-6)
print('Reload prediction parity passed!')
print(pd.DataFrame({'original':before, 'loaded':after}).round(5))


Section 8.2 — Download the artifacts to your local project
What: Download a ZIP containing the models/ directory. Why: Your FastAPI service loads this trained model without needing Colab.

In [ ]:
with zipfile.ZipFile('driftlab_day4_models.zip','w',zipfile.ZIP_DEFLATED) as z:
    for artifact in Path('models').rglob('*'):
        if artifact.is_file():
            z.write(artifact,artifact.as_posix())
print('Created driftlab_day4_models.zip')
try:
    from google.colab import files
    files.download('driftlab_day4_models.zip')
except ImportError:
    print('Not in Colab: retrieve the zip directly from your working directory.')


Section 8.3 — FastAPI Deployment
The service source code lives in api/main.py and src/driftlab/; it must run in a compatible Python environment (such as a Linux runtime). API endpoints: /health, /model/info, /predict.
You do not need to retrain when the API starts. Exporting does not fix Windows application-control restrictions on SciPy; use an approved environment where these packages load.
Try one prediction, then compare its result against the same sensor row in the notebook.

## Section 9 — Day 5: Drift Simulation and Monitoring

This section runs directly in Colab with **no imports from the backend source folder**. It compares an unchanged batch with a simulated torque shift. PSI and total variation describe feature-distribution change, not a proven change in prediction quality.


In [ ]:
# Day 5: self-contained drift calculations (no local package imports required)
import numpy as np
import pandas as pd

def population_stability_index(reference, current, bins=10):
    """PSI using quantile bins from reference only; heuristic, not a significance test."""
    ref = np.asarray(reference, dtype=float)
    cur = np.asarray(current, dtype=float)
    if not np.isfinite(ref).all() or not np.isfinite(cur).all():
        raise ValueError('Drift inputs must contain finite values')
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return np.nan  # insufficient variability for meaningful quantile bins
    edges[0], edges[-1] = -np.inf, np.inf
    ref_count = np.histogram(ref, bins=edges)[0]
    cur_count = np.histogram(cur, bins=edges)[0]
    p = (ref_count + 0.5) / (len(ref) + 0.5 * len(ref_count))
    q = (cur_count + 0.5) / (len(cur) + 0.5 * len(cur_count))
    return float(np.sum((q-p) * np.log(q/p)))

def notebook_drift_report(reference, current, threshold=0.20):
    """Return one drift measurement per numerical feature and Type distribution."""
    if len(reference) < 30 or len(current) < 30:
        raise ValueError('Need at least 30 samples in both batches')
    results = []
    for col in numeric_features:
        psi = population_stability_index(reference[col], current[col])
        results.append({'feature':col, 'metric':'PSI', 'value':psi,
                        'flagged': bool(np.isfinite(psi) and psi >= threshold)})
    categories = ['L', 'M', 'H']
    p = reference['Type'].value_counts(normalize=True).reindex(categories, fill_value=0)
    q = current['Type'].value_counts(normalize=True).reindex(categories, fill_value=0)
    tvd = float(0.5 * np.abs(p - q).sum())
    results.append({'feature':'Type', 'metric':'total variation', 'value':tvd, 'flagged':tvd >= 0.15})
    return pd.DataFrame(results)

# Baseline and incoming batches drawn from held-out subsets for demonstration
reference = X_train.sample(n=min(1000, len(X_train)), random_state=42).copy()
current = X_val.sample(n=min(500, len(X_val)), random_state=42).copy()

print('Unmodified incoming batch:')
display(notebook_drift_report(reference, current))

# Change ONLY a copy: synthetic drift; not an actual machine reading
shifted = current.copy()
shifted['Torque [Nm]'] += 15
print('Incoming batch with simulated +15 Nm torque shift:')
shifted_report = notebook_drift_report(reference, shifted)
display(shifted_report)
print('Flagged features:', shifted_report.loc[shifted_report['flagged'], 'feature'].tolist())
